In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import *

In [0]:
df_brands = spark.table('ecommerce.bronze.brz_brand')
df_brands.show(10)

In [0]:
df_silver_brands = df_brands.withColumn('brand_name', trim(col('brand_name')))
df_silver_brands.show(10)

In [0]:
# Keep only alphanumeric characters in brand_code (removes any special characters / whitespace)
df_silver_brands = df_silver_brands.withColumn('brand_code', regexp_replace(trim(col('brand_code')), '[^A-Za-z0-9]', ''))
df_silver_brands.show(10)

In [0]:
df_silver_brands.select('category_code').distinct().show()

In [0]:
df_silver_brands = df_silver_brands.withColumn("category_code", regexp_replace(col("category_code"), 'BOOKS', 'BKS')).withColumn("category_code", regexp_replace(col("category_code"), 'GROCERY', 'GRCY')).withColumn("category_code", regexp_replace(col("category_code"), 'TOYS', 'TOY'))
df_silver_brands.select('category_code').distinct().show()

In [0]:
df_silver_brands.write.format("delta").mode("overwrite").option("mergeSchema", True).saveAsTable(f"ecommerce.silver.slv_brands")

CATEGORY

In [0]:
df_category = spark.table('ecommerce.bronze.brz_category')
df_category.show(10)

In [0]:
df_duplicates = df_category.groupBy(col("category_code")).count().where(count("*")>1).show()

In [0]:
df_silver_category  = df_category.dropDuplicates()

In [0]:
df_duplicates = df_silver_category.groupBy(col("category_code")).count().where(count("*")>1).show()

In [0]:
df_silver_category = df_silver_category.withColumn("category_code", upper(col("category_code")))
df_silver_category.show(10)

In [0]:
df_silver_category.write.format("delta").mode("overwrite").option("mergeSchema", True).saveAsTable(f"ecommerce.silver.slv_category")

CUSTOMER

In [0]:
df_customer = spark.table('ecommerce.bronze.brz_customer')
df_customer.show(10)

In [0]:
null_count = df_customer.filter(col("customer_id").isNull()).count()
null_count

In [0]:
df_silver_customer = df_customer.dropna(subset=["customer_id"]).fillna("NA", subset="phone")

In [0]:
df_silver_customer.write.format("delta").mode("overwrite").option("mergeSchema", True).saveAsTable(f"ecommerce.silver.slv_customer")

DATE

In [0]:
df_date = spark.table('ecommerce.bronze.brz_date')
df_date.display(10)
df_date

In [0]:
df_date.groupBy("date").count().where(count("*")>1).show()
df_silver_date = df_date.withColumn("date", to_date(col("date"), 'dd-mm-yyyy'))\
    .withColumn("day_name", initcap(col("day_name")))\
        .withColumn("week_of_year", abs(col("week_of_year")))\
            .dropDuplicates(["date"])\
                .withColumn("quarter", concat(lit("Q"), col("quarter"), lit("-"), col("year")))\
                    .withColumnRenamed("week_of_year", "week")

df_silver_date.display()

In [0]:
df_silver_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"ecommerce.silver.slv_date")

In [0]:
spark.sql("DROP TABLE IF EXISTS ecommerce.bronze.brand")

PRODUCTS

In [0]:
df_products = spark.table('ecommerce.bronze.brz_products')
df_products.show(10)

In [0]:
row_count, column_count = df_products.count(), len(df_products.columns)
print(f"row count = {row_count}, column count = {column_count}")

In [0]:
#replacing weight grom xg to just x and converting it into integer
df_silver_products = df_products.withColumn("weight_grams", regexp_replace(col("weight_grams"), "g", "").cast(IntegerType()))\
.withColumn("length_cm", regexp_replace(col("length_cm"), ",", ".").cast(FloatType()))\
.withColumn("category_code", upper(col("category_code")))\
.withColumn("brand_code", upper(col("brand_code")))\
.withColumn("material", regexp_replace(col("material"), "Coton", "Cotton"))\
.withColumn("material", regexp_replace(col("material"), "Ruber", "Rubber"))\
.withColumn("material", regexp_replace(col("material"), "Alumium", "Aluminum"))\
.withColumn("rating_count", abs(col("rating_count")).cast(IntegerType()))
df_silver_products.show()

In [0]:
df_silver_products.write.format("delta").mode("overwrite").option("mergeSchema", True).saveAsTable(f"ecommerce.silver.slv_products")